# GRAPHICAL USER INTERFACE




In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import tensorflow as tf
import numpy as np
import cv2
import gradio as gr
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.applications.efficientnet import preprocess_input

In [5]:
model_path = '/content/drive/MyDrive/Colon_Cancer_Explainability_Guided_Model.h5'

model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("✅ Model loaded successfully")


✅ Model loaded successfully


In [6]:
def grad_cam(model, image):

    img = image.resize((224,224))
    img_array = np.expand_dims(np.array(img), axis=0)
    img_array = preprocess_input(img_array)

    base_model = model.get_layer('efficientnetb0')

    with tf.GradientTape() as tape:
        features = base_model(img_array, training=False)
        tape.watch(features)

        x = model.layers[-4](features)
        x = model.layers[-3](x)
        x = model.layers[-2](x)
        preds = model.layers[-1](x)

        loss = preds[:,0]

    grads = tape.gradient(loss, features)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    heatmap = tf.reduce_sum(features[0] * pooled_grads, axis=-1)
    heatmap = np.maximum(heatmap.numpy(), 0)
    heatmap /= np.max(heatmap) + 1e-8

    heatmap = cv2.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    overlay = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

    return overlay

In [7]:
def predict_with_gradcam(image):

    img = image.resize((224,224))
    img_array = np.expand_dims(np.array(img), axis=0)
    img_array = preprocess_input(img_array)

    score = float(model.predict(img_array)[0][0])
    probability = score * 100

    # Prediction Label
    if score > 0.5:
        label = "🔴 Cancer (Adenocarcinoma)"
        color = "red"
    else:
        label = "🟢 Normal Colon Tissue"
        color = "green"

    # Confidence Level
    if probability >= 90:
        confidence = "Very High ❗❗❗"
    elif probability >= 75:
        confidence = "High ✅"
    elif probability >= 60:
        confidence = "Medium ⚠️"
    else:
        confidence = "Low ❗"

    cam_image = grad_cam(model, image)

    result_html = f"""
    <div style="font-size:26px; font-weight:bold; color:{color};">
    Prediction: {label}
    </div>

    <div style="font-size:20px;">
    Probability: <b>{probability:.2f}%</b>
    </div>

    <div style="font-size:20px;">
    Confidence Level: <b>{confidence}</b>
    </div>
    """

    return result_html, probability, cam_image


In [8]:
interface = gr.Interface(
    fn=predict_with_gradcam,

    inputs=gr.Image(
        type="pil",
        label="Upload Colon Histopathology Image"
    ),

    outputs=[
        gr.HTML(label="Prediction Result"),
        gr.Number(label="Prediction Probability (%)"),
        gr.Image(label="Grad-CAM Visual Explanation")
    ],

    title="Colon Cancer Detection with Explainable AI",

    description="""
Upload a colon histopathology image.
The system predicts whether the colon tissue is Normal or Cancerous and displays
the prediction probability along with confidence level and Grad-CAM visual explanation.
"""
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bbe1c39f0bb44d683c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
